In [1]:
# app.py
"""
Streamlit app: Smart Procurement for Factories
Real-Time Material Price Insights

Dependencies:
    pip install streamlit yfinance pandas plotly

Run:
    streamlit run app.py
"""

import streamlit as st
import pandas as pd
import yfinance as yf
import plotly.express as px
from datetime import datetime, timedelta
import io

st.set_page_config(page_title="Smart Procurement — Real-Time Material Prices", layout="wide")

# --------------------
# Helper / Config
# --------------------
@st.cache_data(ttl=120)  # cache short-term
def fetch_history(ticker: str, period: str = "1mo", interval: str = "1d"):
    """Fetch historical price data for a ticker via yfinance."""
    try:
        df = yf.download(ticker, period=period, interval=interval, progress=False)
        if df.empty:
            return pd.DataFrame()
        df = df.reset_index().rename(columns={"Date": "timestamp"})
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df["close"] = df["Close"]
        df = df[["timestamp", "open", "high", "low", "close", "Volume"]]
        return df
    except Exception as e:
        st.error(f"Error fetching {ticker}: {e}")
        return pd.DataFrame()

@st.cache_data(ttl=60)
def fetch_latest_price(ticker: str):
    """Get most recent price and change info for ticker."""
    try:
        ticker_obj = yf.Ticker(ticker)
        info = ticker_obj.history(period="5d", interval="1d")
        if info.empty:
            return None
        last = info.tail(1)["Close"].iloc[0]
        prev = info.tail(2)["Close"].iloc[0] if len(info) >= 2 else last
        pct = ((last - prev) / prev) * 100 if prev != 0 else 0
        return {"price": float(last), "pct_change": float(pct)}
    except Exception:
        return None

MATERIAL_TICKERS = {
    "Gold (XAU)"        : "GC=F",     # CME Gold Futures
    "Silver (XAG)"      : "SI=F",
    "Crude Oil (WTI)"   : "CL=F",
    "Brent Crude"       : "BZ=F",
    "Natural Gas"       : "NG=F",
    "Copper"            : "HG=F",
    "Aluminium (LME proxy)": "ALI=F",  # sometimes proxies; check for your exchange
    "Corn"              : "ZC=F",
    "Wheat"             : "ZW=F",
    "Soybean"           : "ZS=F",
    "Steel (HRC index)" : "SPLV",      # placeholder - may need custom source
    "Copper (LME)"      : "LCOc1",     # may not work in yfinance; allow custom input
}

# --------------------
# Sidebar: controls
# --------------------
st.sidebar.header("Controls")
st.sidebar.markdown("Choose materials, date range, and alert thresholds.")

# Material selection
selected_materials = st.sidebar.multiselect(
    "Select materials",
    options=list(MATERIAL_TICKERS.keys()),
    default=["Gold (XAU)", "Crude Oil (WTI)"]
)

# Allow custom ticker entry
custom_ticker = st.sidebar.text_input("Add custom ticker (yfinance symbol)", value="", help="Enter a yfinance symbol like 'AAPL' or 'GC=F'. Leave blank if none.")
custom_nickname = st.sidebar.text_input("Nickname for custom ticker (optional)", value="")

# Date range / period
period_map = {
    "7 days": ("7d", "60m"),
    "1 month": ("1mo", "1d"),
    "3 months": ("3mo", "1d"),
    "6 months": ("6mo", "1d"),
    "1 year": ("1y", "1d"),
    "5 years": ("5y", "1wk"),
}
period_label = st.sidebar.selectbox("Time range", list(period_map.keys()), index=1)

# Moving average window
ma_window = st.sidebar.slider("Moving average window (days)", min_value=3, max_value=60, value=14)

# Alert thresholds
st.sidebar.subheader("Alerts (per selected material)")
alert_enable = st.sidebar.checkbox("Enable alerts", value=True)
alert_above = st.sidebar.number_input("Alert if price >= (set per material below)", value=0.0, step=0.01, format="%.2f")
alert_below = st.sidebar.number_input("Alert if price <= (set per material below)", value=0.0, step=0.01, format="%.2f")

refresh_button = st.sidebar.button("Refresh Prices Now")

# CSV upload
upload = st.sidebar.file_uploader("Upload custom price CSV (timestamp,close)", type=["csv"])

# --------------------
# Main UI
# --------------------
st.title("Smart Procurement for Factories — Real-Time Material Price Insights")
st.write("Monitor commodity & material prices, visualize trends, and set procurement alerts.")

left_col, right_col = st.columns([2, 1])

# Prepare the tickers list
tickers = []
material_to_name = {}
for name in selected_materials:
    ticker = MATERIAL_TICKERS.get(name)
    if ticker:
        tickers.append(ticker)
        material_to_name[ticker] = name

# Add custom ticker if provided
if custom_ticker.strip():
    tickers.append(custom_ticker.strip().upper())
    material_to_name[custom_ticker.strip().upper()] = custom_nickname or custom_ticker.strip().upper()

# Show upload preview if CSV provided
if upload is not None:
    try:
        uploaded_df = pd.read_csv(upload, parse_dates=[0])
        uploaded_df.columns = [c.strip() for c in uploaded_df.columns]
        if len(uploaded_df.columns) >= 2:
            uploaded_df = uploaded_df.rename(columns={uploaded_df.columns[0]: "timestamp", uploaded_df.columns[1]: "close"})
        st.sidebar.success("Uploaded CSV parsed (first rows shown).")
        st.sidebar.dataframe(uploaded_df.head())
    except Exception as e:
        st.sidebar.error(f"Unable to parse CSV: {e}")
        uploaded_df = None
else:
    uploaded_df = None

# Fetch latest snapshot table
with right_col:
    st.subheader("Current Snapshot")
    snapshot_rows = []
    for t in tickers:
        data = fetch_latest_price(t)
        if data:
            snapshot_rows.append({
                "ticker": t,
                "name": material_to_name.get(t, t),
                "price": round(data["price"], 4),
                "% change": round(data["pct_change"], 3)
            })
        else:
            snapshot_rows.append({"ticker": t, "name": material_to_name.get(t, t), "price": None, "% change": None})
    snapshot_df = pd.DataFrame(snapshot_rows)
    if snapshot_df.empty:
        st.write("No tickers selected or data unavailable.")
    else:
        st.dataframe(snapshot_df.set_index("name"))

    # Alerts display
    if alert_enable and not snapshot_df.empty:
        st.markdown("**Alerts**")
        alerted = False
        for _, row in snapshot_df.iterrows():
            price = row["price"]
            if price is None:
                continue
            # apply simple alert logic: if alert_above/below set (non-zero)
            if alert_above and price >= alert_above:
                st.warning(f"{row['name']}: price {price} ≥ alert_above {alert_above}")
                alerted = True
            if alert_below and alert_below and price <= alert_below:
                st.warning(f"{row['name']}: price {price} ≤ alert_below {alert_below}")
                alerted = True
        if not alerted:
            st.success("No active alerts right now.")

# Main charts and analysis
with left_col:
    st.subheader("Price Trends & Analysis")
    period_str, interval = period_map[period_label]

    # If CSV uploaded, show that as one dataset
    if uploaded_df is not None:
        df = uploaded_df.copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp")
        df["MA"] = df["close"].rolling(window=ma_window, min_periods=1).mean()
        fig = px.line(df, x="timestamp", y=["close", "MA"], labels={"value": "Price", "timestamp": "Time"}, title="Uploaded Price Series")
        st.plotly_chart(fig, use_container_width=True)
        st.download_button("Download uploaded as CSV", df.to_csv(index=False).encode("utf-8"), file_name="uploaded_prices.csv")
    else:
        # Plot each ticker time series on same chart
        all_plots = []
        combined_df = pd.DataFrame()
        for t in tickers:
            hist = fetch_history(t, period=period_str, interval=interval)
            if hist.empty:
                continue
            hist = hist.assign(ticker=t)
            hist["name"] = material_to_name.get(t, t)
            hist["MA"] = hist["close"].rolling(window=ma_window, min_periods=1).mean()
            combined_df = pd.concat([combined_df, hist], ignore_index=True)

        if combined_df.empty:
            st.info("No historical data available for the selected tickers/time range. Try adding a custom ticker or reduce time range.")
        else:
            # interactive selector
            material_names = combined_df["name"].unique().tolist()
            chosen = st.multiselect("Pick materials to visualize (from fetched data)", material_names, default=material_names[:2])

            if chosen:
                plot_df = combined_df[combined_df["name"].isin(chosen)]
                # Plot close price
                fig = px.line(plot_df, x="timestamp", y="close", color="name", title="Close Price Over Time", labels={"close": "Close Price", "timestamp": "Time"})
                st.plotly_chart(fig, use_container_width=True)

                # Plot moving averages for each
                fig2 = px.line(plot_df, x="timestamp", y="MA", color="name", title=f"{ma_window}-day Moving Average")
                st.plotly_chart(fig2, use_container_width=True)

                # Show table with latest values and percent change over period
                latest = plot_df.sort_values("timestamp").groupby("name").tail(1).set_index("name")
                earliest = plot_df.sort_values("timestamp").groupby("name").head(1).set_index("name")
                summary = pd.DataFrame({
                    "latest_price": latest["close"],
                    "earliest_price": earliest["close"],
                })
                summary["pct_change_over_period"] = ((summary["latest_price"] - summary["earliest_price"]) / summary["earliest_price"]) * 100
                st.subheader("Summary")
                st.dataframe(summary.round(4))

                # Export CSV of combined plots
                csv = plot_df[["timestamp", "name", "close", "MA"]].to_csv(index=False).encode("utf-8")
                st.download_button("Download plotted data as CSV", csv, file_name="material_prices.csv")
            else:
                st.info("Select at least one material to visualize.")

# Bottom: procurement suggestions + notes
st.markdown("---")
st.header("Procurement Signals (simple heuristics)")
st.write("""
This section shows **simple, example heuristics** you can extend for procurement decisions.
Examples:
- If price is trending up > X% over the last N days, consider hedging or delaying purchases.
- If price dips below historical moving average by > Y%, consider buying.
""")

heuristics_col1, heuristics_col2 = st.columns(2)
with heuristics_col1:
    trend_period_days = st.number_input("Trend lookback (days)", min_value=3, max_value=365, value=30)
    trend_threshold_pct = st.number_input("Trigger if trend change >= (%)", min_value=0.1, max_value=100.0, value=5.0)

with heuristics_col2:
    buy_below_ma_pct = st.number_input("Buy if price <= MA by (%)", min_value=0.1, max_value=100.0, value=3.0)
    show_signals = st.button("Evaluate procurement signals")

if show_signals:
    if 'combined_df' not in locals() or combined_df.empty:
        st.warning("No market data to evaluate signals. Select materials and refresh.")
    else:
        signals = []
        now = combined_df["timestamp"].max()
        lookback_start = now - pd.Timedelta(days=int(trend_period_days))
        for name in combined_df["name"].unique():
            sub = combined_df[combined_df["name"] == name].sort_values("timestamp")
            recent = sub[sub["timestamp"] >= lookback_start]
            if recent.shape[0] < 2:
                continue
            start_price = recent.iloc[0]["close"]
            end_price = recent.iloc[-1]["close"]
            pct = ((end_price - start_price) / start_price) * 100
            latest_ma = sub.iloc[-1]["MA"]
            latest_price = sub.iloc[-1]["close"]
            signal = {"material": name, "trend_pct": round(pct, 3), "latest_price": round(latest_price, 4), "latest_MA": round(latest_ma, 4)}
            if pct >= trend_threshold_pct:
                signal["action"] = "Consider hedging / buy smaller lots (uptrend)"
            elif (latest_price <= latest_ma * (1 - buy_below_ma_pct / 100.0)):
                signal["action"] = "Consider buying (price below MA)"
            else:
                signal["action"] = "Hold / Monitor"
            signals.append(signal)
        if signals:
            st.table(pd.DataFrame(signals))
        else:
            st.info("No clear signals found for selected materials.")

# Footer / tips
st.markdown("---")
st.markdown("**Tips & next steps:**\n- Replace or augment yfinance with a dedicated commodities/market data API (Metals-API, Quandl, Bloomberg) for enterprise-grade feeds.\n- Add authenticated APIs and a job to poll prices at intervals and persist to a DB (Postgres, TimescaleDB) for real realtime.\n- Integrate notifications via email/SMS/Slack when alerts trigger.\n- Add currency conversion if your procurement is priced in different currencies.")

# Manual refresh action
if refresh_button:
    st.experimental_rerun()


2026-02-01 10:43:01.573 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 10:43:01.573 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-01 10:43:01.573 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-01 10:43:01.573 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 10:43:02.643 
  command:

    streamlit run d:\Procrument System\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-01 10:43:02.643 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-01 10:43:02.643 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runnin

In [3]:
!pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ---------------------------------------- 0.0/949.2 kB ? eta -:--:--
     ---------------------------------------- 949.2/949.2 kB 14.6 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 14.4 MB/s  0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15616 sha256=2291f9055e8111ff806fe2c6d8b3337bd525d565236cb6525b568ae71812ec35
  Stored in directory: c:\users

  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'multitasking'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [5]:
!pip install datetime

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   ---------------------------------------- 0/2 [zope.interface]
   -------------------- ------------------- 1/2 [datetime]
   -------------------- ------------------- 1/2 [datetime]
   --------------------

In [4]:
!pip install plotly

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   --------------- ------------------------ 3.9/9.8 MB 19.5 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.8 MB 16.1 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.8 MB 15.0 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 13.0 MB/s  0:00:00
